# Kidney outcomes analysis

Original MED264 analysis, prepared for version control.

Run only in an authorized All of Us Researcher Workbench with Registered Tier v7 access. Outputs and notebook metadata have been cleared for distribution. Review `docs/analysis-notes.md` for fixes and unresolved methodological limitations. The corrected analysis has not been rerun on participant data. Never commit executed participant-data outputs.


In [ ]:
import os
import pandas as pd
import numpy as np

if not os.environ.get("WORKSPACE_CDR"):
    raise RuntimeError("Run this notebook in an authorized All of Us Researcher Workbench.")


In [ ]:
import pandas
import os

# This query represents dataset "dataset_with_diseases" for domain "person" and was generated for All of Us Registered Tier Dataset v7
dataset_33371088_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (201820, 201826)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 ) 
                    AND age_at_event >= 18 ) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN (442793, 443732, 4193704)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 
                    UNION
                    DISTINCT SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN (1567956, 44833365)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 0 
                            AND is_selectable = 1) 
                        AND is_standard = 0 ) 
                    AND age_at_event >= 18 ) criteria ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (43526465)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (44785829)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (45774751)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (793293)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (45774435)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (1583722)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (40170911)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (44506754)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (793143)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) 
                        AND cb_search_person.person_id IN (SELECT
                            criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN (46236952, 3053283, 40764999, 40771922) 
                            AND is_standard = 1 )) criteria ) )"""

dataset_33371088_person_df = pandas.read_gbq(
    dataset_33371088_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_33371088_person_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "dataset_with_diseases" for domain "drug" and was generated for All of Us Registered Tier Dataset v7
dataset_33371088_drug_sql = """
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_exposure.drug_type_concept_id,
        d_type.concept_name as drug_type_concept_name,
        d_exposure.stop_reason,
        d_exposure.refills,
        d_exposure.quantity,
        d_exposure.days_supply,
        d_exposure.sig,
        d_exposure.route_concept_id,
        d_route.concept_name as route_concept_name,
        d_exposure.lot_number,
        d_exposure.visit_occurrence_id,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_value,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        d_exposure.route_source_value,
        d_exposure.dose_unit_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                        WHERE
                            concept_id IN (1583722, 40170911, 43526465, 44506754, 44785829, 45774435, 45774751, 793143, 793293)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))  
                    AND (d_exposure.PERSON_ID IN (SELECT
                        distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (201820, 201826)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 ) 
                            AND age_at_event >= 18 ) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            person_id IN (SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (442793, 443732, 4193704)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) 
                                AND is_standard = 1 
                            UNION
                            DISTINCT SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (1567956, 44833365)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 0 
                                    AND is_selectable = 1) 
                                AND is_standard = 0 ) 
                            AND age_at_event >= 18 ) criteria ) 
                            AND cb_search_person.person_id IN (SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (43526465)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (44785829)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (45774751)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (793293)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (45774435)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (1583722)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (40170911)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (44506754)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (793143)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria ) 
                                AND cb_search_person.person_id IN (SELECT
                                    criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN (46236952, 3053283, 40764999, 40771922) 
                                    AND is_standard = 1 )) criteria ) )
                        )
                    ) d_exposure 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept 
                        ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_type 
                        ON d_exposure.drug_type_concept_id = d_type.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_route 
                        ON d_exposure.route_concept_id = d_route.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                        ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_visit 
                        ON v.visit_concept_id = d_visit.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_source_concept 
                        ON d_exposure.drug_source_concept_id = d_source_concept.concept_id"""

dataset_33371088_drug_df = pandas.read_gbq(
    dataset_33371088_drug_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_33371088_drug_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "dataset_with_diseases" for domain "measurement" and was generated for All of Us Registered Tier Dataset v7
dataset_33371088_measurement_sql = """
    SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        measurement.measurement_datetime,
        measurement.measurement_type_concept_id,
        m_type.concept_name as measurement_type_concept_name,
        measurement.operator_concept_id,
        m_operator.concept_name as operator_concept_name,
        measurement.value_as_number,
        measurement.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        measurement.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        measurement.range_low,
        measurement.range_high,
        measurement.visit_occurrence_id,
        m_visit.concept_name as visit_occurrence_concept_name,
        measurement.measurement_source_value,
        measurement.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary,
        measurement.unit_source_value,
        measurement.value_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (1619025, 3004249, 3012888, 3022318, 3025315, 3027018, 3031203, 3036277, 3038553, 36303797, 36306178, 36660257, 40759207, 40764999, 40765148)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                measurement.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (201820, 201826)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 ) 
                            AND age_at_event >= 18 ) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            person_id IN (SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (442793, 443732, 4193704)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) 
                                AND is_standard = 1 
                            UNION
                            DISTINCT SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (1567956, 44833365)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 0 
                                    AND is_selectable = 1) 
                                AND is_standard = 0 ) 
                            AND age_at_event >= 18 ) criteria ) 
                            AND cb_search_person.person_id IN (SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (43526465)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (44785829)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (45774751)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (793293)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (45774435)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (1583722)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (40170911)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (44506754)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (793143)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria ) 
                                AND cb_search_person.person_id IN (SELECT
                                    criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN (46236952, 3053283, 40764999, 40771922) 
                                    AND is_standard = 1 )) criteria ) )
                        )
                    ) measurement 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_standard_concept 
                        ON measurement.measurement_concept_id = m_standard_concept.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_type 
                        ON measurement.measurement_type_concept_id = m_type.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_operator 
                        ON measurement.operator_concept_id = m_operator.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_value 
                        ON measurement.value_as_concept_id = m_value.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_unit 
                        ON measurement.unit_concept_id = m_unit.concept_id 
                LEFT JOIn
                    `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                        ON measurement.visit_occurrence_id = v.visit_occurrence_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_visit 
                        ON v.visit_concept_id = m_visit.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_source_concept 
                        ON measurement.measurement_source_concept_id = m_source_concept.concept_id"""

dataset_33371088_measurement_df = pandas.read_gbq(
    dataset_33371088_measurement_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_33371088_measurement_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "dataset_with_diseases" for domain "condition" and was generated for All of Us Registered Tier Dataset v7
dataset_33371088_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (134057, 316139, 320128, 440383)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (201820, 201826)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 ) 
                            AND age_at_event >= 18 ) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            person_id IN (SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (442793, 443732, 4193704)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) 
                                AND is_standard = 1 
                            UNION
                            DISTINCT SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (1567956, 44833365)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 0 
                                    AND is_selectable = 1) 
                                AND is_standard = 0 ) 
                            AND age_at_event >= 18 ) criteria ) 
                            AND cb_search_person.person_id IN (SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (43526465)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (44785829)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (45774751)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (793293)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (45774435)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (1583722)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (40170911)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (44506754)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (793143)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria ) 
                                AND cb_search_person.person_id IN (SELECT
                                    criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN (46236952, 3053283, 40764999, 40771922) 
                                    AND is_standard = 1 )) criteria ) )
                        )
                    ) c_occurrence 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                        ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
                        ON c_occurrence.condition_type_concept_id = c_type.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                        ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
                        ON v.visit_concept_id = visit.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
                        ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
                        ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

dataset_33371088_condition_df = pandas.read_gbq(
    dataset_33371088_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_33371088_condition_df.head(5)

In [ ]:
print(dataset_33371088_person_df.columns)
print(dataset_33371088_drug_df.columns)
print(dataset_33371088_measurement_df.columns)
print(dataset_33371088_person_df.shape)
print(dataset_33371088_drug_df.shape)
print(dataset_33371088_measurement_df.shape)

### The following cells deal with patient dataframe

In [ ]:
from datetime import datetime
dataset_person_concise = dataset_33371088_person_df[["person_id", "gender", "date_of_birth", "race", "ethnicity", "sex_at_birth"]].copy()
# Fixed reference date preserves the historical analysis; see limitations.
reference_date = pandas.Timestamp('2024-11-20', tz='UTC')
dataset_person_concise["age"] = (reference_date - dataset_person_concise['date_of_birth']).dt.days // 365
dataset_person_concise

### The following cells deal with drug dataframe

In [ ]:
dataset_33371088_drug_df["standard_concept_name"].unique()

# Change the drug name to universal format
dataset_drug_process = dataset_33371088_drug_df.copy()

# Using regex to replace the complex drug name to the simple name
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*canagliflozin.*$', value='canagliflozin', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*dapagliflozin.*$', value='dapagliflozin', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*empagliflozin.*$', value='empagliflozin', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*ertugliflozin.*$', value='ertugliflozin', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*dulaglutide.*$', value='dulaglutide', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*exenatide.*$', value='exenatide', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*liraglutide.*$', value='liraglutide', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*lixisenatide.*$', value='lixisenatide', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*semaglutide.*$', value='semaglutide', regex=True)

print(dataset_drug_process["standard_concept_name"].unique())
dataset_drug_process.head(10)

# Find the first exposure of the drug. 
patients = dataset_drug_process["person_id"].unique()

# Add drug type
drug_type_dict = {'ertugliflozin':"SGLT2i", 'exenatide':"GLP-1RA", 'liraglutide':"GLP-1RA", 'canagliflozin':"SGLT2i", 'empagliflozin':"SGLT2i",
 'dulaglutide':"GLP-1RA", 'dapagliflozin':"SGLT2i", 'semaglutide':"GLP-1RA", 'lixisenatide':"GLP-1RA",}

dataset_drug_process["initial_exposure"] = ""
dataset_drug_process["drug_type"] = dataset_drug_process["standard_concept_name"].map(drug_type_dict)
for patient in patients:
    patient_data_frame = dataset_drug_process.query("person_id == @patient")
    drug_exposure_list = patient_data_frame["drug_exposure_start_datetime"].to_list()
    earliest_exposure_time = min(drug_exposure_list)

    dataset_drug_process.loc[dataset_drug_process['person_id'] == patient, 'initial_exposure'] = earliest_exposure_time



dataset_drug_process["initial_exposure"] = pandas.to_datetime(dataset_drug_process["initial_exposure"])

In [ ]:
# Convert to weeks and do some calculations
dataset_drug_process["relative_drug_exposure_start_datetime_by_weeks"] = (dataset_drug_process["drug_exposure_start_datetime"] - dataset_drug_process["initial_exposure"]).dt.days / 7
dataset_drug_process["relative_drug_exposure_end_datetime_by_weeks"] = (dataset_drug_process["drug_exposure_end_datetime"] - dataset_drug_process["initial_exposure"]).dt.days / 7
dataset_drug_process["relative_exposure_week"] = dataset_drug_process["relative_drug_exposure_end_datetime_by_weeks"] - dataset_drug_process["relative_drug_exposure_start_datetime_by_weeks"]


In [ ]:
# remove unuseful columns
dataset_drug_concise = dataset_drug_process[["person_id", "standard_concept_name", "drug_type", "initial_exposure", "relative_drug_exposure_start_datetime_by_weeks", "relative_drug_exposure_end_datetime_by_weeks", "relative_exposure_week"]]
dataset_drug_concise = dataset_drug_concise.rename(columns={"standard_concept_name": "drug_name"})
dataset_drug_concise.to_csv("dataset_drug_concise.csv")
dataset_drug_concise

In [ ]:
patient_drug_type = dataset_drug_concise[["person_id", "drug_type"]].drop_duplicates()
type_1 = set(patient_drug_type.query("drug_type == 'SGLT2i'")["person_id"].to_list())
type_2 = set(patient_drug_type.query("drug_type == 'GLP-1RA'")["person_id"].to_list())


from matplotlib_venn import venn2, venn3
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
venn2([type_1, type_2], set_labels=('SGLT2i', 'GLP-1RA'))
plt.title("Venn Diagram of SGLT2i and GLP-1RA")

# Filter the patients only have one medicine 
single_drug_patient_SGLT2i = type_1.difference(type_2)
single_drug_patient_GLP = type_2.difference(type_1)
len(single_drug_patient_SGLT2i), len(single_drug_patient_GLP)

single_drug_patient = single_drug_patient_SGLT2i.union(single_drug_patient_GLP)

### The following cells deal with measurement dataframe

In [ ]:
dataset_33371088_measurement_df["standard_concept_name"].value_counts()

In [ ]:
dataset_33371088_measurement_df["standard_concept_name"].unique()
dataset_measurement_process = dataset_33371088_measurement_df.copy()
# Unify Glomerular filteration rate. 

dataset_measurement_process["standard_concept_name"].unique()

kept_element = ['Systolic blood pressure', 'Body weight',
       'Body mass index (BMI) [Ratio]', 'Body height',
       'Diastolic blood pressure', 'Heart rate',
       'Glomerular filtration rate/1.73 sq M.predicted [Volume Rate/Area] in Serum, Plasma or Blood by Creatinine-based formula (CKD-EPI)',
       'PhenX - hip circumference protocol 020801',
       'Adult Waist Circumference Protocol', 'Heart rate rhythm',
       'Blood pressure panel']

dataset_measurement_process = dataset_measurement_process[dataset_measurement_process["standard_concept_name"].isin(kept_element)]
dataset_measurement_process["standard_concept_name"] = dataset_measurement_process["standard_concept_name"].replace(to_replace=r'^.*Glomerular.*$', value='GFR', regex=True)

dataset_measurement_process["standard_concept_name"].unique()

In [ ]:
# Get first drug exposure time
first_exposure = dataset_drug_concise[["person_id", "initial_exposure"]]
first_exposure = first_exposure.drop_duplicates()
first_exposure

# Merge it with measurement dataframe
pandas.set_option('display.max_columns', None)
dataset_measurement_process = dataset_measurement_process.merge(first_exposure, on = "person_id")
dataset_measurement_process["relative_measurement_datetime_by_weeks"] = (dataset_measurement_process["measurement_datetime"]-dataset_measurement_process["initial_exposure"]).dt.days / 7
dataset_measurement_concise = dataset_measurement_process[["person_id", "standard_concept_name", "measurement_datetime", "value_as_number", "unit_source_value", "initial_exposure", "relative_measurement_datetime_by_weeks"]]
dataset_measurement_concise = dataset_measurement_concise.rename(columns={"standard_concept_name": "measurement"})
dataset_measurement_concise

# Get the one close to initial exposure

In [ ]:
patient_with_GFR = dataset_measurement_concise[["person_id", "measurement"]]
patient_with_GFR = patient_with_GFR.query("measurement == 'GFR'")
patient_with_GFR = patient_with_GFR.drop_duplicates()
patient_with_GFR = patient_with_GFR["person_id"].to_list()

In [ ]:
# patient 
# time 0(first exposure), follow them for a certain period of time (36 months). Have the Glomerular measurement 
# censor: they drop or leave the study, or they didn't develop the events (no CDK measurement), 
# if you stop using the drug for more than 6 month, they got censor
# Event: they have the kidney disease (40% or more reduction in EGFR from the baseline(first measurement nearest to first exposure))

def check_whether_drop(weeks, threshold, end_month):
    sorted_number = sorted(weeks)
    sorted_number.append(end_month)
    
    for i in range(len(sorted_number)-1):
        if sorted_number[i + 1] - sorted_number[i] > threshold:
            return True
    return False

single_drug_patient_list = list(single_drug_patient)
measurement_list = ['Systolic blood pressure', 'Body weight',
       'Body mass index (BMI) [Ratio]', 'Body height',
       'Diastolic blood pressure', 'Heart rate',
       'PhenX - hip circumference protocol 020801',
       'Adult Waist Circumference Protocol', 'Heart rate rhythm',
       'Blood pressure panel', 'GFR']
n = 0

event_dataset = pandas.DataFrame(columns=['person_id', 'initial_GFR', 'final_GFR', 'Event20', 'Event40','final_measure_time', 'drug_type'])

for patient_id in single_drug_patient_list:
    # Check whether need to be filtered (drop or leave the study)
    three_year_to_weeks = 52*3
    six_month = 26
    patient_drug_dataset = dataset_drug_concise.query("person_id == @patient_id")
    patient_drug_start_time = patient_drug_dataset["relative_drug_exposure_start_datetime_by_weeks"].to_list()

    if patient_id not in patient_with_GFR:
        continue
#     if check_whether_drop(patient_drug_start_time, six_month, three_year_to_weeks):
#         continue
    n = n + 1
    drug_type = "SGLT2i"
    if patient_id in list(single_drug_patient_GLP):
        drug_type = "GLP-1RA"

    
    patient_measurement = dataset_measurement_concise.query("person_id == @patient_id & measurement == 'GFR'")
    measurement_time = patient_measurement["relative_measurement_datetime_by_weeks"].to_list()
    intial_measurement_time = min(measurement_time, key =abs)
    final_measurement_time = min(measurement_time, key=lambda x: abs(x - three_year_to_weeks))
#     patient_measurement = dataset_measurement_concise.query("person_id == @patient_id")
    initial_measurement_dataset = patient_measurement.query("relative_measurement_datetime_by_weeks == @intial_measurement_time")
    final_measurement_dataset = patient_measurement.query("relative_measurement_datetime_by_weeks == @final_measurement_time")
    
    initial_GFR = initial_measurement_dataset.query("measurement == 'GFR'")["value_as_number"].iloc[0]
    final_GFR = final_measurement_dataset.query("measurement == 'GFR'")["value_as_number"].iloc[0]
    Event20 = False

    if initial_GFR * 0.8 >= final_GFR:
        Event20 = True
        
    Event40 = False
    
    if initial_GFR * 0.6 >= final_GFR:
        Event40 = True
    event_dataset.loc[len(event_dataset)] = [patient_id, initial_GFR, final_GFR, Event20, Event40, final_measurement_time, drug_type]
    
event_dataset

    


    



In [ ]:
# event_counts = event_dataset['Event'].value_counts()

# event_counts.plot.bar()
# plt.show()

# drug_counts = event_dataset['drug_type'].value_counts()

# drug_counts.plot.bar()
# plt.show()

# # plot = event_dataset.plot.pie(y='Event', figsize=(5, 5))

In [ ]:
# event_counts

In [ ]:
final_dataset = event_dataset.merge(dataset_person_concise, on = "person_id", how = "left")

In [ ]:
final_dataset

### Process disease condition

In [ ]:
condition_dataset = dataset_33371088_condition_df.copy()

condition_dataset["standard_concept_name"] = condition_dataset["standard_concept_name"].replace(to_replace=r'^.*hypertens.*$', value='hypertension', regex=True)
condition_dataset["standard_concept_name"] = condition_dataset["standard_concept_name"].replace(to_replace=r'^.*heart failure.*$', value='heart failure', regex=True)
condition_dataset["standard_concept_name"] = condition_dataset["standard_concept_name"].replace(to_replace=r'^.*cardiovascular.*$', value='cardiovascular', regex=True)
condition_dataset["standard_concept_name"] = condition_dataset["standard_concept_name"].replace(to_replace=r'^.*depress.*$', value='depression', regex=True)
need = ["hypertension", "heart failure", "cardiovascular", "depression"]
condition_dataset = condition_dataset.query("standard_concept_name in @need")
condition_dataset.drop_duplicates()
condition_dataset




In [ ]:
condition_dataset_final = pandas.DataFrame(columns=['person_id', 'hypertension', 'heart_failure', 'cardiovascular', 'depression'])
id_list = final_dataset["person_id"].to_list()
for patient_id in id_list:
    person_id = patient_id
    query_data = condition_dataset.query("person_id == @patient_id")


    hypertension = ((query_data['standard_concept_name'] == 'hypertension') & (query_data['person_id'] == patient_id)).any()
    heart_failure = ((query_data['standard_concept_name'] == 'heart failure') & (query_data['person_id'] == patient_id)).any()
    cardiovascular = ((query_data['standard_concept_name'] == 'cardiovascular') & (query_data['person_id'] == patient_id)).any()
    depression = ((query_data['standard_concept_name'] == 'depression') & (query_data['person_id'] == patient_id)).any()
    
    condition_dataset_final.loc[len(condition_dataset_final)] = [person_id, hypertension, heart_failure, cardiovascular, depression]

condition_dataset_final

In [ ]:
final_dataset = final_dataset.merge(condition_dataset_final, on = "person_id", how = "left")
# drug_name = dataset_drug_concise[["person_id", "drug_name"]]
# drug_name = drug_name.drop_duplicates()
# final_dataset = final_dataset.merge(drug_name, on = "person_id", how = "left")
final_dataset

In [ ]:
final_dataset["hypertension"].value_counts()

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.regression.mixed_linear_model import MixedLM
import pandas as pd
import numpy as np

# Check full dataset characteristics
print("Dataset shape:", final_dataset.shape)
print("\nColumns in dataset:", final_dataset.columns.tolist())

# Check missing values
print("\nMissing values:\n", final_dataset.isnull().sum())

# Basic descriptive statistics for continuous variables
print("\nBasic statistics:")
print(final_dataset[['initial_GFR', 'final_GFR', 'age', 'final_measure_time']].describe())

# Distribution of key variables
print("\nDrug type distribution:")
print(final_dataset['drug_type'].value_counts(normalize=True))

print("\nGender distribution:")
print(final_dataset['gender'].value_counts(normalize=True))

print("\nRace distribution:")
print(final_dataset['race'].value_counts(normalize=True))

# Create binary variables for the model
final_dataset['drug_type_binary'] = (final_dataset['drug_type'] == 'SGLT2i').astype(int)
final_dataset['gender_binary'] = (final_dataset['gender'] == 'Male').astype(int)
final_dataset['race_binary'] = (final_dataset['race'] == 'White').astype(int)

# Check the time variable distribution
print("\nTime variable summary:")
print(final_dataset['final_measure_time'].describe())

# Check number of measurements per person
measurements_per_person = final_dataset.groupby('person_id').size()
print("\nMeasurements per person:")
print(measurements_per_person.describe())

# Save the processed dataset
# Dataset export omitted from the repository version.


In [ ]:
bool_columns = ['hypertension', 'heart_failure', 'cardiovascular', 'depression']

# Now let's try the analysis again
# Create clean dataset
clean_dataset = final_dataset.copy()

# Remove missing GFR values
clean_dataset = clean_dataset.dropna(subset=['initial_GFR', 'final_GFR'])

# Recode gender (excluding the third category)
clean_dataset = clean_dataset[clean_dataset['gender'].isin(['Male', 'Female'])]
clean_dataset['gender_binary'] = (clean_dataset['gender'] == 'Male').astype(int)

# Recode race (binary: White vs Non-White)
clean_dataset['race_binary'] = (clean_dataset['race'] == 'White').astype(int)

# Create drug type binary
clean_dataset['drug_type_binary'] = (clean_dataset['drug_type'] == 'SGLT2i').astype(int)

# Standardize continuous variables
clean_dataset['time_std'] = (clean_dataset['final_measure_time'] - clean_dataset['final_measure_time'].mean()) / clean_dataset['final_measure_time'].std()
clean_dataset['age_std'] = (clean_dataset['age'] - clean_dataset['age'].mean()) / clean_dataset['age'].std()

# Calculate GFR change
clean_dataset['gfr_change'] = clean_dataset['final_GFR'] - clean_dataset['initial_GFR']

# Convert boolean columns to int
for col in bool_columns:
    clean_dataset[col] = clean_dataset[col].astype(int)

# Check data types after conversion
print("\nData types after conversion:")
print(clean_dataset[['gfr_change', 'drug_type_binary', 'time_std', 'age_std', 'gender_binary', 
                    'race_binary', 'hypertension', 'heart_failure', 'cardiovascular', 
                    'depression']].dtypes)

# Create X matrix for regression
X = clean_dataset[['drug_type_binary', 'time_std', 'age_std', 'gender_binary', 
                  'race_binary', 'hypertension', 'heart_failure', 'cardiovascular', 
                  'depression']]
X = sm.add_constant(X)

# Create y vector
y = clean_dataset['gfr_change']

# Fit model
model = sm.OLS(y, X)
results = model.fit()

# Print results
print("\nRegression Results:")
print(results.summary())

# Basic descriptive statistics
print("\nMean GFR change by drug type:")
print(clean_dataset.groupby('drug_type')['gfr_change'].agg(['count', 'mean', 'std']))

In [ ]:
# 1. Check correlations between predictors
correlation_matrix = clean_dataset[['drug_type_binary', 'time_std', 'age_std', 
                                  'gender_binary', 'race_binary', 'hypertension', 
                                  'heart_failure', 'cardiovascular', 'depression']].corr()
print("Correlation Matrix:")
print(correlation_matrix)

# 2. Create interaction terms
clean_dataset['drug_time_interaction'] = clean_dataset['drug_type_binary'] * clean_dataset['time_std']
clean_dataset['drug_age_interaction'] = clean_dataset['drug_type_binary'] * clean_dataset['age_std']

# 3. Fit improved model with interactions
X_interaction = sm.add_constant(clean_dataset[[
    'drug_type_binary',
    'time_std',
    'age_std',
    'gender_binary',
    'race_binary',
    'hypertension',
    'depression',  # Removing heart_failure and cardiovascular due to potential collinearity
    'drug_time_interaction',
    'drug_age_interaction'
]])

y = clean_dataset['gfr_change']
model_interaction = sm.OLS(y, X_interaction)
results_interaction = model_interaction.fit()

print("\nImproved Model Results:")
print(results_interaction.summary())

# 4. Calculate mean GFR change by drug type and time periods
clean_dataset['time_quartile'] = pd.qcut(clean_dataset['final_measure_time'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
time_drug_summary = clean_dataset.groupby(['time_quartile', 'drug_type'])['gfr_change'].agg(['count', 'mean', 'std']).round(2)
print("\nGFR Change by Time Period and Drug Type:")
print(time_drug_summary)

# 5. Basic statistics by drug type
drug_stats = clean_dataset.groupby('drug_type').agg({
    'initial_GFR': ['mean', 'std'],
    'final_GFR': ['mean', 'std'],
    'gfr_change': ['mean', 'std', 'count']
}).round(2)

print("\nDetailed Statistics by Drug Type:")
print(drug_stats)

In [ ]:
# 1. Create baseline GFR categories
clean_dataset['baseline_gfr_cat'] = pd.cut(clean_dataset['initial_GFR'],
                                         bins=[0, 30, 60, 90, np.inf],
                                         labels=['Severe', 'Moderate', 'Mild', 'Normal'])

# Analysis by baseline GFR
baseline_analysis = clean_dataset.groupby(['baseline_gfr_cat', 'drug_type'])['gfr_change'].agg([
    'count', 'mean', 'std'
]).round(2)

print("GFR Change by Baseline Category:")
print(baseline_analysis)

# 2. Test for different effects in subgroups
# For patients with lower baseline GFR
low_gfr = clean_dataset[clean_dataset['initial_GFR'] < 60]
X_low = sm.add_constant(low_gfr[['drug_type_binary', 'time_std', 'age_std', 
                                'gender_binary', 'race_binary', 'hypertension', 'depression']])
y_low = low_gfr['gfr_change']
model_low = sm.OLS(y_low, X_low).fit()

print("\nModel Results for Low Baseline GFR (<60):")
print(model_low.summary().tables[1])

# 3. Compare early vs late effects
clean_dataset['early_period'] = clean_dataset['final_measure_time'] <= clean_dataset['final_measure_time'].median()
early_vs_late = clean_dataset.groupby(['early_period', 'drug_type'])['gfr_change'].agg([
    'count', 'mean', 'std'
]).round(2)

print("\nEarly vs Late Period Comparison:")
print(early_vs_late)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Baseline GFR Category Plot
plt.figure(figsize=(10, 6))
baseline_plot = clean_dataset.groupby(['baseline_gfr_cat', 'drug_type'])['gfr_change'].mean().unstack()
baseline_plot.plot(kind='bar', yerr=clean_dataset.groupby(['baseline_gfr_cat', 'drug_type'])['gfr_change'].std().unstack())
plt.title('Mean GFR Change by Baseline Category and Drug Type')
plt.xlabel('Baseline GFR Category')
plt.ylabel('Mean GFR Change')
plt.xticks(rotation=45)
plt.legend(title='Drug Type')
plt.tight_layout()
plt.show()

# 2. Time-based changes
plt.figure(figsize=(10, 6))
sns.boxplot(data=clean_dataset, x='time_quartile', y='gfr_change', hue='drug_type')
plt.title('GFR Change Distribution by Time Quartile and Drug Type')
plt.xlabel('Time Quartile')
plt.ylabel('GFR Change')
plt.tight_layout()
plt.show()

# 3. Additional analysis for specific subgroups
# Focus on moderate to severe CKD
ckd_subset = clean_dataset[clean_dataset['initial_GFR'] < 60].copy()
print("\nDetailed Analysis for CKD Patients (GFR < 60):")
print(ckd_subset.groupby(['drug_type', 'hypertension'])['gfr_change'].agg(['count', 'mean', 'std']).round(2))

In [ ]:
# Check correlations
correlations = clean_dataset[['heart_failure', 'cardiovascular', 'hypertension']].corr()
print("\nCorrelations between cardiovascular conditions:")
print(correlations)

# Check distributions
print("\nDistribution of conditions:")
print("Heart Failure:")
print(clean_dataset['heart_failure'].value_counts())
print("\nCardiovascular:")
print(clean_dataset['cardiovascular'].value_counts())
print("\nHypertension:")
print(clean_dataset['hypertension'].value_counts())

# Let's try model with each condition separately
from statsmodels.stats.outliers_influence import variance_inflation_factor

def check_vif(data, features):
    X = data[features]
    vif_data = pd.DataFrame()
    vif_data["Variable"] = features
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

In [ ]:
# Revised model without heart_failure and cardiovascular
X_revised = sm.add_constant(clean_dataset[[
    'drug_type_binary',
    'time_std',
    'age_std',
    'gender_binary',
    'race_binary',
    'hypertension',
    'depression'
]])

revised_model = sm.OLS(clean_dataset['gfr_change'], X_revised).fit()
print(revised_model.summary())

In [ ]:
# Calculate frequencies
print("Event20 Distribution:")
print(clean_dataset['Event20'].value_counts())
print("\nEvent40 Distribution:")
print(clean_dataset['Event40'].value_counts())

# Look at events by drug type
print("\nEvent20 by Drug Type:")
print(pd.crosstab(clean_dataset['drug_type'], clean_dataset['Event20']))
print("\nEvent40 by Drug Type:")
print(pd.crosstab(clean_dataset['drug_type'], clean_dataset['Event40']))

# Calculate percentages of events by drug type
print("\nPercentage of Event20 by Drug Type:")
print(pd.crosstab(clean_dataset['drug_type'], clean_dataset['Event20'], normalize='index') * 100)
print("\nPercentage of Event40 by Drug Type:")
print(pd.crosstab(clean_dataset['drug_type'], clean_dataset['Event40'], normalize='index') * 100)

# Check events by GFR categories
print("\nEvents by Baseline GFR Category:")
print(pd.crosstab([clean_dataset['baseline_gfr_cat'], clean_dataset['drug_type']], 
                  [clean_dataset['Event20'], clean_dataset['Event40']]))

In [ ]:
# Logistic regression for Event20
import statsmodels.api as sm
X = sm.add_constant(clean_dataset[[
    'drug_type_binary',
    'time_std',
    'age_std',
    'gender_binary',
    'race_binary',
    'hypertension',
    'depression'
]])

# Model for Event20
model_event20 = sm.Logit(clean_dataset['Event20'], X).fit()
print("\nEvent20 Model Results:")
print(model_event20.summary2().tables[1])

# Model for Event40
model_event40 = sm.Logit(clean_dataset['Event40'], X).fit()
print("\nEvent40 Model Results:")
print(model_event40.summary2().tables[1])